In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# get some real data to play with for gradient descent
data = pd.read_csv('https://github.com/dvasiliu/AML/blob/main/Data%20Sets/Advertising.csv?raw=true')

In [ ]:
data

,Unnamed: 0,TV,Radio,Newspaper,Sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9
...,...,...,...,...,...
195,196,38.2,3.7,13.8,7.6
196,197,94.2,4.9,8.1,9.7
197,198,177.0,9.3,6.4,12.8
198,199,283.6,42.0,66.2,25.5


## The Main Goal
 Determine the weights for TV, Radio and Newspaper advertising to predict the Sales. We want to minimize the sum of squared errors by applying gradient descent.

In [20]:
y = data['Sales'].values
x = data.loc[:,['TV','Radio','Newspaper']].values

We want to predict $y$ by using
$$w_0\cdot 1 + w_1\cdot \text{TV} + w_2\cdot \text{Radio} +w_3\cdot \text{Newspaper}$$

In [3]:
def zscore(x):
  return (x-np.mean(x))/np.std(x)

In [21]:
x_scaled = zscore(x)

In [22]:
# we can use a bias term or we can center y
# to generate a bias term we need to augment x by including a columns of 1s
x_aug = np.column_stack([np.ones(len(x)),x_scaled])

In [ ]:
x_aug

In [23]:
# we need MSE and its gradient
def MSE(w,x,y):
  return np.mean((y-x@w)**2)

In [24]:
# we must compute the gradient of MSE
def gradient(w,x,y):
  n = len(x)
  return 2/n*(y-x@w)@(-x)

In [25]:
w = [1,2,3,4]
# all we need to update the weights in the direction of the negative gradient
lr = 0.02 # in reality this value needs to be guessed or found independently
max_iter = 10000

In [ ]:
for i in range(max_iter):
  w_new = w - lr*gradient(w,x_aug,y)
  if np.mean(abs(w-w_new))<1e-6:
    break
  if (i+1)%50==0:
    print(f"At the iteration {i+1} the Mean Squared Error is {MSE(w_new,x_aug,y)}")
  w = w_new

## The Mini-Batch SGD

In [28]:
np.array_split(np.random.permutation(range(len(x))),15)

[array([ 70,  16,  60, 154,  79, 190, 161, 172, 142, 128,  20,  38,  23,
         21]),
 array([198, 196,  52, 157, 107, 181,  93,  94, 131, 180, 145, 141,  45,
         43]),
 array([113, 168,  95,  58,  81, 156,  40, 112, 110, 169, 195,  51, 184,
         50]),
 array([  7, 150, 101,  77,  65,  37, 136,  62,  42, 115, 176, 155,  25,
        153]),
 array([  4, 186,  64, 125,  59,  32, 151, 175,  89,   3,   9,  19,  48,
         67]),
 array([146,  36,  22,  46, 139, 160,  30, 130, 152,  29, 119, 182, 192]),
 array([100, 121,  18,  54,  12,  75,  97, 118,   6,  39,  73, 171, 104]),
 array([ 56, 126,  96,  28,  78,  99,  47,  57,  86, 162, 105, 174, 111]),
 array([ 92, 143, 167, 199, 114, 144, 120, 123, 103,  82,  10,   5, 179]),
 array([129,  13, 164, 117, 188, 116,   8, 132, 147, 138, 158,  34,  26]),
 array([124, 197, 177,  33,  74,  14, 165,  53, 122,  72,  24, 108,  76]),
 array([187,  91,  17, 173, 166,  15,  84, 194, 178, 163, 148,   1, 159]),
 array([ 87,  80, 185,  27,  90, 19

In [31]:
w = [1,2,3,4]
batch_size = 15
epochs = 10000
lr = 0.03

In [ ]:
for i in range(epochs):
  for batch_ind in np.array_split(np.random.permutation(range(len(x))),batch_size):
    xs = x_aug[batch_ind]
    ys = y[batch_ind]
    w_new = w - lr*gradient(w,xs,ys)
    if np.mean(abs(w-w_new))<1e-6:
      break
    if (i+1)%100==0:
      print(f"At the iteration {i+1} the Mean Squared Error is {MSE(w_new,x_aug,y)}")
    w = w_new

## RMSPROP

In [40]:
w = [1,2,3,4]
batch_size = 15
epochs = 10000
eta = 0.1
gama = 0.5
epsilon = 0.01
s = 0
path = [w]

In [ ]:
for i in range(epochs):
  for batch_ind in np.array_split(np.random.permutation(range(len(x))),batch_size):
    xs = x_aug[batch_ind]
    ys = y[batch_ind]
    lr = eta/np.sqrt(s+epsilon)
    g = gradient(w,xs,ys)
    w_new = w - lr*g
    path.append(w_new)
    s = gama*s + (1-gama)*sum(g**2)
    if np.mean(abs(w-w_new))<1e-6:
      break
    if (i+1)%100==0:
      print(f"At the iteration {i+1} the Mean Squared Error is {MSE(w_new,x_aug,y)}")
    w = w_new

In [54]:
path_MSE = []
for i in range(len(path)):
  path_MSE.append(MSE(path[i],x_aug,y))

In [55]:
# the best MSE
np.min(path_MSE)

2.7841402903921666

In [56]:
# the best set of weights
path[np.argmin(path_MSE)]

array([18.54813981,  3.5121079 , 14.46399424, -0.09038976])